# Preprocessing for Real data

- Create mask from thermal video

In [ ]:
import sys
sys.path.append("../")

import os
workspace_path = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
print(workspace_path)
sys.path.append(workspace_path)

import numpy as np
import matplotlib.pyplot as plt
import cv2
import matplotlib.patches as patches
from ipywidgets import interact, widgets
import pickle
import zstandard as zstd
import io

from lib.utils_real import *

# Fix: KeyError 'modifiers' due to ipympl/matplotlib version mismatch
# handle_button_press etc. are class-level aliases to _handle_mouse,
# so all aliases must be patched individually.
import matplotlib.backends.backend_webagg_core as _bwc
_orig_handle_mouse = _bwc.FigureCanvasWebAggCore._handle_mouse
def _patched_handle_mouse(self, event):
    event.setdefault('modifiers', [])
    return _orig_handle_mouse(self, event)

for _alias in (
    '_handle_mouse',
    'handle_button_press',
    'handle_button_release',
    'handle_dblclick',
    'handle_figure_enter',
    'handle_figure_leave',
    'handle_motion_notify',
    'handle_scroll',
):
    setattr(_bwc.FigureCanvasWebAggCore, _alias, _patched_handle_mouse)

#### File name settings

In [ ]:
video_filename = "paintedbear.npz"

## Pre-processing

### Load Video

In [ ]:
def load_recording_2d(filepath):
    """
    Load frames as (T, H, W) from either old or new data format.

    Old bunny format (.npz, key="frame_recording"):
        (num_views, T, H, W) -> take [0] -> (T, H, W)

    Old thermal_video format (.npz, key="frames"):
        (T, H, W) -> as-is

    New format (.npz.zst, key="raw_thr_frames_A"):
        (T, H+2, W, 1)
          -> squeeze last dim    -> (T, H+2, W)
          -> crop 2 header rows  -> (T, H, W)   [row0=metadata, row1=zeros]
    """
    if filepath.endswith('.zst'):
        with open(filepath, 'rb') as f:
            dctx = zstd.ZstdDecompressor()
            decompressed = dctx.decompress(f.read())
        data = np.load(io.BytesIO(decompressed), allow_pickle=True)
        frames = data['raw_thr_frames_A']   # (T, H+2, W, 1)
        frames = frames[..., 0]             # (T, H+2, W)
        frames = frames[:, 2:, :]          # (T, H, W)  crop header rows
    else:
        data = np.load(filepath, allow_pickle=True)
        if 'frame_recording' in data:
            frames = data['frame_recording']
            if frames.ndim == 4:            # (num_views, T, H, W)
                frames = frames[0]          # (T, H, W)
        else:
            frames = data['frames']         # (T, H, W)
    return frames

frame_recording = load_recording_2d(
    os.path.join(workspace_path, "DATA", "obj_study_raw", video_filename)
)
print(frame_recording.shape)

In [ ]:
plt.close('all')
with plt.ioff():
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(1, 1, 1)
frame_min = frame_recording.min()
frame_max = frame_recording.max()

def update(val):
    ax.clear()
    frame_dis = 255 * (frame_recording[int(val)] - frame_min * 0.8) / (frame_max - frame_min)
    frame_dis = cv2.applyColorMap(frame_dis.astype(np.uint8), cv2.COLORMAP_MAGMA)
    ax.imshow(frame_dis)
    ax.set_title(
        'Heat flow at t = {}, \n max-c={:.2f}, min-c={:.2f}'.format(
            int(val),
            np.max(frame_recording[int(val)]),
            np.min(frame_recording[int(val)])
        )
    )
    ax.axis('off')
    fig.canvas.draw()

interact(update, val=widgets.IntSlider(min=0, max=frame_recording.shape[0]-1, step=10, value=0))
display(fig)

### Mask Extraction

In [ ]:
HEAT_START_FRAME = 5
print(frame_recording.shape)

# frame_recording[0]（first frame）
bg_frame = frame_recording[0].copy().astype(float)  # (H, W)
avg_diff_frame = None

for i in range(HEAT_START_FRAME, frame_recording.shape[0] - 1):
    curr = frame_recording[i + 1].astype(float)
    diff = np.abs(curr - bg_frame)
    if avg_diff_frame is None:
        avg_diff_frame = diff
    else:
        avg_diff_frame += diff

avg_diff_frame = avg_diff_frame / (frame_recording.shape[0] - HEAT_START_FRAME)

avg_diff_frame_dis = cv2.normalize(avg_diff_frame, None, 0, 255,cv2.NORM_MINMAX, cv2.CV_8U)
plt.imshow(avg_diff_frame, cmap='hot')
plt.axis('off')
plt.show()

In [ ]:
%matplotlib widget

# --- base mask: 2D bool is easier to work with ---
H, W = avg_diff_frame.shape[:2]
frame_mask = np.ones((H, W), dtype=bool)

# For vis (contrast enhanced)
norm8 = cv2.normalize(avg_diff_frame, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
norm8 = clahe.apply(norm8)
norm_avg_frame = norm8[:, :, None]  # (H,W,1)

def make_display_img():
    return (norm_avg_frame * frame_mask[:, :, None]).astype(np.uint8)

avg_diff_frame_dis = make_display_img()

# --- polygon points (added by clicking) ---
poly_points = []  # [(x,y), (x,y), ...]

# Vis
fig, ax = plt.subplots(figsize=(10, 8))
fig.suptitle("Left click: add points | r: admit | b: undo | c: clear current | Esc: quit")
img_disp = ax.imshow(avg_diff_frame_dis, cmap='gray', vmin=0, vmax=255)
ax.set_title("Masked Frame")
ax.set_axis_off()

poly_patch = patches.Polygon([[0, 0]], closed=True, fill=False, edgecolor='red', linewidth=2, visible=False)
ax.add_patch(poly_patch)
pts_scatter = ax.scatter([], [], s=20)

def update_display():
    global avg_diff_frame_dis, poly_points
    avg_diff_frame_dis = make_display_img()
    img_disp.set_data(avg_diff_frame_dis)

    if len(poly_points) >= 2:
        poly_patch.set_xy(np.array(poly_points))
        poly_patch.set_visible(True)
    else:
        poly_patch.set_visible(False)

    if len(poly_points) > 0:
        xs = [p[0] for p in poly_points]
        ys = [p[1] for p in poly_points]
        pts_scatter.set_offsets(np.c_[xs, ys])
    else:
        pts_scatter.set_offsets(np.empty((0, 2)))

    fig.canvas.draw_idle()

def on_click(event):
    global poly_points
    if event.inaxes != ax:
        return
    if event.button != 1:
        return
    x = int(event.xdata)
    y = int(event.ydata)
    if 0 <= x < W and 0 <= y < H:
        poly_points.append((x, y))
        print(f"Add point: ({x},{y})  [total={len(poly_points)}]")
        update_display()

def admit_polygon():
    global frame_mask, poly_points
    if len(poly_points) < 3:
        print("Need at least 3 points to make a polygon.")
        return
    pts = np.array(poly_points, dtype=np.int32).reshape((-1, 1, 2))
    mask_poly = np.zeros((H, W), dtype=np.uint8)
    cv2.fillPoly(mask_poly, [pts], 1)
    mask_poly = mask_poly.astype(bool)
    frame_mask = np.logical_and(frame_mask, mask_poly)
    print(f"Admit polygon with {len(poly_points)} points. (mask AND applied)")
    poly_points = []
    update_display()

def on_key(event):
    global poly_points
    key = event.key
    if key == 'escape':
        print("Esc pressed. Close figure.")
        plt.close(fig)
    elif key == 'c':
        print("Clear CURRENT polygon points (mask stays).")
        poly_points = []
        update_display()
    elif key == 'b':
        if len(poly_points) > 0:
            removed = poly_points.pop()
            print(f"Undo last point: {removed}  [total={len(poly_points)}]")
            update_display()
        else:
            print("No points to undo.")
    elif key == 'r':
        admit_polygon()

fig.canvas.mpl_connect('button_press_event', on_click)
fig.canvas.mpl_connect('key_press_event', on_key)

plt.show()

print("Final mask shape:", frame_mask.shape)

plt.figure()
plt.title("Final Frame Mask")
plt.imshow(frame_mask, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
frame_mask = get_largest_closed_mask(frame_mask, method=cv2.RETR_TREE)
plt.imshow(frame_mask)
plt.show()

### Save Mask

In [ ]:
mask_filename = video_filename.split('.')[0] + '_mask.pkl'
mask_filepath = os.path.join(workspace_path, 'DATA', 'obj_study_proc', mask_filename)
os.makedirs(os.path.join(workspace_path, 'DATA', 'obj_study_proc'), exist_ok=True)
with open(mask_filepath, 'wb') as f:
    pickle.dump({'mask': frame_mask, 'mean': 0.0}, f)
cv2.imwrite(mask_filepath.replace('.pkl', '.png'), frame_mask * 255)
print("Saved mask to", mask_filepath)
